## Interaction Embedding

This notebook produces a fixed-size vector representation for each student–exercise interaction in the MIAAM dataset.

**Pipeline:**
1. Load `maths_data_filtered.parquet` (student attempt records).
2. Filter out interactions whose exercise has no compressed screenshot, so that all remaining records have visual content available.
3. Map string `user_id` values to contiguous integer indices (`user_id_int`).
4. Encode interactions in chunks of 1 000 000 rows using `encode_interactions` from `attempt_embedding` — each interaction is represented as a `Float32` vector derived from the attempt context.
5. Save each chunk to `data/interactions_embedded_part_{i}.parquet` with a globally-unique `interaction_id` offset applied across chunks.

**Output:** one embedding per interaction, saved across multiple part files, suitable as sequential input features for downstream student-modelling or knowledge-tracing models.

In [1]:

import polars as pl
import json
import pathlib
import os
import numpy as np
from exercise_embedding import encode_all
from attempt_embedding import encode_interactions
from tqdm import tqdm
import gc
import torch



DATASET = pathlib.Path("../../../MIAAM")
SAVE_FOLDER = pathlib.Path("../data")

interactions = pl.read_parquet(DATASET / "data/maths_data_filtered.parquet")

/home/guglielmo/Projects/stundetsTracker/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Filter exercises without a screenshot

Some exercises in `maths_exercises_table.parquet` have no corresponding screenshot in `data/screenshots/compressed/`. They are removed from both `exercises` and `interactions` before any further processing, so that all remaining exercises have visual content available for multimodal models.

In [2]:
SCREENSHOTS = DATASET / "data/screenshots/compressed"
screenshot_ids = {
    f.stem
    for source_dir in SCREENSHOTS.iterdir()
    for f in source_dir.iterdir()
    if f.suffix == ".png"
}

interactions = interactions.filter(pl.col("exercise_id").is_in(list(screenshot_ids)))
print(f"\nAfter filtering: {interactions.shape[0]} attempts, {interactions['exercise_id'].n_unique()} unique exercises")


After filtering: 6263671 attempts, 6936 unique exercises


In [3]:
user_id_map = (
    interactions.select("user_id")
    .unique()
    .sort("user_id")
    .with_row_index("user_id_int")
)
interactions = interactions.join(user_id_map, on="user_id", how="left")
del user_id_map

In [4]:

CHUNK_SIZE = 1000000
n_rows = len(interactions)
n_chunks = (n_rows + CHUNK_SIZE - 1) // CHUNK_SIZE

part_paths = []

for i in range(n_chunks):
    part_path = SAVE_FOLDER / f"interactions_embedded_part_{i}.parquet"

    if part_path.exists():
        print(f"Part {i+1}/{n_chunks} already exists, skipping.")
        part_paths.append(part_path)
        continue

    start = i * CHUNK_SIZE
    chunk = interactions.slice(start, CHUNK_SIZE)
    print(f"Encoding part {i+1}/{n_chunks} (rows {start}–{start + len(chunk) - 1})...")

    chunk_embeddings = encode_interactions(chunk)
    # Offset so interaction_id is globally unique across all chunks
    chunk_embeddings = chunk_embeddings.with_columns(
        (pl.col("interaction_id") + start).alias("interaction_id")
    )
    chunk_embeddings.write_parquet(part_path)
    part_paths.append(part_path)

    del chunk, chunk_embeddings
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  Saved and freed part {i+1}.")

print(f"Done — {n_chunks} parts saved to {SAVE_FOLDER}")

Part 1/7 already exists, skipping.
Part 2/7 already exists, skipping.
Part 3/7 already exists, skipping.
Part 4/7 already exists, skipping.
Part 5/7 already exists, skipping.
Part 6/7 already exists, skipping.
Part 7/7 already exists, skipping.
Done — 7 parts saved to ../data
